### GM Probability Pipeline

Following SUITPy tutorial: isolates human cerebellum and outputs grey matter probability in functional regions defined in the Nettekoven atlas (asymmtric 32-region).

Inputs: T1w anatomical, cerebellar grey matter

**tutorial: ** 
https://suitpy.readthedocs.io/en/latest/index.html

In [1]:
# Import packages
from nilearn import plotting as npl
import SUITPy as suit
import SUITPy.atlas as atlas
import nibabel as nib
import ants
import matplotlib.pyplot as plt

In [ ]:
t1_img = 'JHU_2713_W0_T1.nii' # replace with path to file
gm_img = 'c1JHU_2713_W0_T1.nii'

In [ ]:
# 1. isolation module

"""
`isolate` function generates an isolation MASK for cerebellum; registration doen by function
input: T1-weighted scan
output: cerebellar mask (as ANTsImage)
"""
mask = suit.isolate(t1_img)
mask_img = 'JHU_2713_W0_T1_cerebellum_dseg.nii.gz'

"""
# visualize results
img = nib.load('JHU_2713_W0_T1.nii')
mask = nib.load('JHU_2713_W0_T1_cerebellum_dseg.nii.gz')
plotting.plot_roi(mask, img)
"""

preprocessing JHU_2713_W0_T1.nii
isolating cerebellum using UNet model
postprocessing
saving results to JHU_2713_W0_T1_cerebellum_dseg.nii.gz


"\n# visualize results\nimg = nib.load('JHU_2713_W0_T1.nii')\nmask = nib.load('JHU_2713_W0_T1_cerebellum_dseg.nii.gz')\nplotting.plot_roi(mask, img)\n"

In [3]:
# 2. normalization module

"""
Input: cerebellar mask (from isolation); source image
Output: (1) normalized cerebellum: anatomical in SUIT space; (2) deformation field
Normalizes to the SUIT cerebellar template
"""

results = suit.normalize(source_file = t1_img,
                         mask_file = mask_img,
                         write_jacobian_determinant=True, # save jac det
                         verbose = 1)

deformation_image = results['fwd_deformation']
normalized_image = results['normalized_image']
jac_det = results['jacobian_determinant']

Normalizing JHU_2713_W0_T1 to tpl-SUIT_T1w.nii.gz
Saving the normalized image into JHU_2713_W0_T1_space-SUIT.nii.gz
Saving deformation field into JHU_2713_W0_T1_to-SUIT_mode-image_xfm.nii.gz
Saving the log Jacobian determinant to JHU_2713_W0_T1_to-SUIT_mode-image_detJ.nii.gz


In [4]:
# 3. reslice image

output_img = suit.reslice_image(source_image = gm_img, # grey matter
                                deformation = deformation_image,
                                mask = mask_img)

#nib.save(output_img, 'img1.nii')

"""
# optional plotting

npl.plot_anat(output_img, cut_coords=(-1, -58, -36)) # specify 3-dim coordinates

"""

'\n# optional plotting\n\nnpl.plot_anat(output_img, cut_coords=(-1, -58, -36)) # specify 3-dim coordinates\n\n'

In [5]:
# 4. dataframe for resliced image (choose an atlas)

# in this case, using Nettekoven_2024 with asymmetric 32-region map (for hand function region, M3)
atlas.fetch_atlas('Nettekoven_2024')
df = atlas.summarize_data(output_img, space = 'SUIT', # use same space as normalization
                     stats = ['nanmean'],
                     atlas = 'Nettekoven_2024', maps = 'atl-NettekovenAsym32')

df.rename(columns={'nanmean': 'GM_prob'}, inplace = True)

In [6]:
df

,image,image_name,frame,region,regionname,size,atlas,map,space,GM_prob
0,1,<nibabel_image>,0,1,M1L,1699.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.696478
1,1,<nibabel_image>,0,2,M2L,4252.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.693611
2,1,<nibabel_image>,0,3,M3L,9115.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.729658
3,1,<nibabel_image>,0,4,M4L,2329.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.524752
4,1,<nibabel_image>,0,5,A1L,717.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.779350
5,1,<nibabel_image>,0,6,A2L,9619.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.650041
6,1,<nibabel_image>,0,7,A3L,3474.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.695339
7,1,<nibabel_image>,0,8,D1L,5153.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.626351
8,1,<nibabel_image>,0,9,D2L,2640.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.593039
9,1,<nibabel_image>,0,10,D3L,9941.0,Nettekoven_2024,atl-NettekovenAsym32,SUIT,0.678269


In [8]:
df.to_csv('JHU2713.csv', index = True)

In [ ]:
# df.to_csv('.csv', mode = 'a', index = True, header = False)